# Fig. S2A: dataset and pseudotime QC

Sub-panels under `Fig2/supp_1/<panel>/`. S1a per-sample composition + the induced-vs-sample
confound that requires pseudobulk; S1b per-lineage pseudotime QC. (No human PAS-distance panel
here; organoid and mouse peak-to-AATAAA distances are computed in
`analysis/pas_distance/261007-pas_hexamer_distance_organoid_vs_mouse.ipynb`.)


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "environment" / "apa-python.yml").is_file()
    )
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()

FIG = "Fig2/supp_1"
H5 = f"{DATA_ROOT}/3utr/organoids/knoblich/cellranger/aggregated_annotated_fix_filtered_normalized_dpt.h5ad"
adata = sc.read_h5ad(H5, backed="r")
obs = adata.obs.copy()
print("cells:", len(obs), "| samples:", obs["sample"].nunique())

In [ ]:
# S1b - per-lineage pseudotime QC
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, 720 / 300))
for lin, lab, col in [
    ("Excitatory neurons", "EXC", S.paper_palette_lineage["excitatory"]),
    ("Inhibitory neurons", "INH", S.paper_palette_lineage["inhibitory"]),
]:
    cc = "dpt_pseudotime_excitatory" if lin == "Excitatory neurons" else "dpt_pseudotime_inhibitory"
    pt = obs.loc[obs.lineage == lin, cc].dropna().values
    ax.hist(
        pt,
        bins=40,
        range=(0, 1),
        color=col,
        alpha=0.6,
        edgecolor="none",
        label=f"{lab} (n={len(pt):,})",
    )
ax.set_xlabel("diffusion pseudotime", fontsize=S.FONT_LABEL)
ax.set_ylabel("cells", fontsize=S.FONT_LABEL)
ax.set_title("per-lineage pseudotime coverage (controls + induced)", fontsize=S.FONT_ANNOT, pad=3)
ax.legend(fontsize=S.FONT_LEGEND, frameon=False, loc="upper center")
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(labelsize=S.FONT_TICK)
fig.subplots_adjust(left=0.12, right=0.96, top=0.91, bottom=0.15)
S.save_and_show(fig, "fig_s1b_pseudotime_qc", figure=FIG)

## S1b size variants (quarter- and eighth-height, half width)


In [ ]:
# --- S1b size variants: quarter-height + eighth-height (half width) ---
from matplotlib.ticker import MaxNLocator


def _build_s1b(figsize, tiny):
    fig, ax = plt.subplots(figsize=figsize)
    for lin, lab, col in [
        ("Excitatory neurons", "EXC", S.paper_palette_lineage["excitatory"]),
        ("Inhibitory neurons", "INH", S.paper_palette_lineage["inhibitory"]),
    ]:
        cc = (
            "dpt_pseudotime_excitatory"
            if lin == "Excitatory neurons"
            else "dpt_pseudotime_inhibitory"
        )
        pt = obs.loc[obs.lineage == lin, cc].dropna().values
        ax.hist(
            pt,
            bins=30 if tiny else 40,
            range=(0, 1),
            color=col,
            alpha=0.6,
            edgecolor="none",
            label=f"{lab} (n={len(pt):,})",
        )
    ax.set_xlabel("pseudotime" if tiny else "diffusion pseudotime", fontsize=S.FONT_LABEL)
    ax.set_ylabel("cells", fontsize=S.FONT_LABEL)
    ax.set_xticks([0, 0.5, 1.0])
    ax.yaxis.set_major_locator(MaxNLocator(nbins=3 if tiny else 4))
    ax.legend(
        fontsize=S.FONT_LEGEND,
        frameon=False,
        loc="upper right",
        handlelength=1.0,
        handletextpad=0.4,
    )
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=S.FONT_TICK)
    if not tiny:
        ax.set_title("per-lineage pseudotime coverage", fontsize=S.FONT_ANNOT, pad=3)
        fig.subplots_adjust(left=0.12, right=0.97, top=0.90, bottom=0.16)
    else:
        fig.subplots_adjust(left=0.13, right=0.98, top=0.93, bottom=0.32)
    return fig


S.save_and_show(
    _build_s1b((S.HALF_W_IN, S.QUARTER_H_IN), False),
    "fig_s1b_pseudotime_qc__quarter_half",
    figure=FIG,
)
S.save_and_show(
    _build_s1b((S.HALF_W_IN, S.EIGHTH_H_IN), True),
    "fig_s1b_pseudotime_qc__eighth_half",
    figure=FIG,
)